# MVSR – Vorbereitung Einheit 3
## Kanten, Hough-Transformation und einfache Bildmerkmale

**Bearbeitungszeit:** ca. 30–45 Minuten

Dieses Notebook bereitet die Inhalte der dritten MVSR-Einheit praktisch vor. Im Mittelpunkt stehen **Kanten und geometrische Bildmerkmale**, die aus einem Kamerabild extrahiert werden können.

Ausgehend von einem realen Testbild untersuchen Sie zunächst den Bildgradienten mit Sobel und den Canny Edge Detector. Anschließend werden Linien und Kreise mit der Hough-Transformation gesucht. Zum Abschluss wird mit Template Matching eine einfache Methode zur Suche eines bekannten Bildausschnitts betrachtet.

Für dieses Binder-Notebook wird **C++17 mit OpenCV 4.6** verwendet.

### Lernziele

Nach Bearbeitung dieses Notebooks können Sie ...

- erklären, warum Kanten für die Reduktion von Bildinformation nützlich sein können,
- den Bildgradienten in x- und y-Richtung interpretieren,
- Sobel- und Canny-Kantendetektion mit OpenCV anwenden,
- den Einfluss von Canny-Schwellwerten untersuchen,
- das Grundprinzip der Hough-Transformation für Linien und Kreise erklären,
- Linien und Kreise mit OpenCV detektieren,
- ein Template in einem größeren Bild mit Template Matching lokalisieren,
- Grenzen einfacher kanten- und templatebasierter Verfahren einschätzen.

> **Hinweis:** Ziel ist nicht, bereits alle Verfahren vollständig zu beherrschen. Das Notebook soll die zentralen Konzepte vor der Vorlesung praktisch erfahrbar machen.

### Hinweise zur Verwendung dieses Notebooks

Dieses Notebook wird über **MyBinder** direkt im Browser ausgeführt. Der vorhandene C++-Code kann unmittelbar in den Notebook-Zellen bearbeitet und ausgeführt werden; eine zusätzliche lokale Installation ist dafür nicht erforderlich.

Das Notebook besteht aus Text- und Codezellen. Die Textzellen enthalten kurze Erklärungen und Aufgabenstellungen, die C++-Codezellen können direkt ausgeführt und verändert werden.

Eine Codezelle wird über den **Play-Button** oder mit **Shift + Enter** ausgeführt. Die Ausgabe erscheint anschließend direkt unterhalb der jeweiligen Zelle.

Arbeiten Sie das Notebook am besten **von oben nach unten** durch, da spätere Codezellen teilweise auf zuvor definierten Variablen und Funktionen aufbauen.

Bei Aufgaben mit `TODO` sollen Sie den vorhandenen Code selbstständig ergänzen oder verändern.

C++ wird mit **xeus-cling** inkrementell ausgeführt. Dadurch können bereits deklarierte Variablen bei einer erneuten Ausführung derselben Zelle zu einer Fehlermeldung führen. Wo dies für eine Übungszelle relevant ist, werden lokale Lambda-Funktionen verwendet. Falls nötig, starten Sie den Kernel neu und führen Sie die Zellen erneut von oben nach unten aus.

> **Wichtig:** Änderungen innerhalb einer Binder-Sitzung werden nicht dauerhaft im GitHub-Repository gespeichert.

## 0. Setup

Die Binder-Umgebung ist bereits mit **xeus-cling** und **OpenCV 4.6** vorbereitet.

> **Binder-/Notebook-Hinweis:** Die folgenden `#pragma cling`-Anweisungen sind ein Workaround für den interaktiven C++-Kernel. Sie teilen `xeus-cling` mit, wo die OpenCV-Header und die kompilierten Bibliotheken liegen. In einem normalen lokalen C++-Projekt wird OpenCV stattdessen beim Kompilieren bzw. über das Build-System, z.B. mit CMake, eingebunden und gelinkt. Im eigentlichen C++-Quellcode genügt dann üblicherweise `#include <opencv2/opencv.hpp>`.

In [ ]:
#include <iostream>
#include <iomanip>
#include <vector>
#include <string>
#include <cmath>
#include <fstream>
#include <cstdlib>
#include <algorithm>

// OpenCV-Pfade für Binder / Ubuntu
#pragma cling add_include_path("/usr/include/opencv4")
#pragma cling add_library_path("/usr/lib/x86_64-linux-gnu")

// Benötigte OpenCV-Bibliotheken laden
#pragma cling load("opencv_core")
#pragma cling load("opencv_imgproc")
#pragma cling load("opencv_imgcodecs")
#pragma cling load("opencv_features2d")
#pragma cling load("opencv_calib3d")

#include <opencv2/opencv.hpp>

std::cout << "OpenCV-Version: " << CV_VERSION << std::endl;
std::cout << "C++ Standard: " << __cplusplus << std::endl;

### Hilfsfunktion zur Darstellung

OpenCV verwendet in einem lokalen C++-Programm normalerweise `cv::imshow()` zur Bilddarstellung. Dabei wird ein eigenes Fenster geöffnet, z.B.

```cpp
cv::imshow("Bild", image);
cv::waitKey(0);
```

In Binder läuft das Notebook jedoch im Browser und besitzt keine normale Desktop-GUI.

> **Binder-/Notebook-Hinweis:** Die folgende Hilfsfunktion ist daher ein Workaround für die Browser-Umgebung. Sie kodiert eine `cv::Mat` als PNG und übergibt sie an die Rich-Display-Funktion des Jupyter-Kernels. Damit ein Bild dargestellt wird, muss `show_image(...)` als **letzte Expression einer Zelle ohne Semikolon** stehen.

In [ ]:
#include "nlohmann/json.hpp"
#include "xtl/xbase64.hpp"

namespace nl = nlohmann;

namespace mvsr
{
    struct NotebookImage
    {
        std::string png_data;

        explicit NotebookImage(const cv::Mat& image)
        {
            std::vector<unsigned char> buffer;
            cv::imencode(".png", image, buffer);

            png_data.assign(
                reinterpret_cast<const char*>(buffer.data()),
                buffer.size()
            );
        }
    };

    nl::json mime_bundle_repr(const NotebookImage& image)
    {
        auto bundle = nl::json::object();
        bundle["image/png"] = xtl::base64encode(image.png_data);
        return bundle;
    }
}

mvsr::NotebookImage show_image(const cv::Mat& image)
{
    return mvsr::NotebookImage(image);
}

### Hilfsfunktion zum Laden externer Testbilder

Einige der folgenden Beispiele verwenden offizielle OpenCV-Testbilder aus GitHub. In der Python-/Colab-Version werden diese mit `!wget` geladen.

> **Binder-/Notebook-Hinweis:** In einer C++-Zelle steht die IPython-Schreibweise `!wget` nicht zur Verfügung. Deshalb wird der Download hier über `std::system()` und das Kommandozeilenprogramm `wget` ausgeführt. In einem normalen lokalen C++-Projekt würde man die benötigten Testbilder üblicherweise lokal ablegen und anschließend direkt mit `cv::imread()` laden.

In [ ]:
bool download_file(
    const std::string& url,
    const std::string& output_file
)
{
    std::string command =
        "wget -q \"" + url + "\" -O \"" + output_file + "\"";

    int result = std::system(command.c_str());

    if (result != 0)
    {
        std::cerr << "Download fehlgeschlagen: "
                  << url << std::endl;
        return false;
    }

    return true;
}

## 1. Testbild laden

Wie in der vorherigen Vorbereitung verwenden wir ein **reales Testbild** aus dem GitHub-Repository der Lehrveranstaltung.

Die niedrige Auflösung reicht für die folgenden Versuche aus und hält die Berechnungen übersichtlich. Bei Interesse kann alternativ die hochauflösende Variante verwendet werden.

Alternativ können Sie auch eigene Bilder im Repository bzw. in der Binder-Sitzung ablegen und laden. Stellenweise müssen bei anderen Testbildern die Parameter in den Codeblöcken angepasst werden.

> **Binder-/Notebook-Hinweis:** MyBinder klont beim Start das vollständige GitHub-Repository. Das Testbild kann daher direkt aus dem Ordner `test_images` geladen werden. Da das Notebook selbst in einem Unterordner liegen kann, werden mehrere mögliche relative Pfade geprüft.

In [ ]:
cv::Mat image;
std::string image_path;

// Niedrige Auflösung
std::string image_filename = "fhtw_logo_low_res.png";

// Alternativ: hohe Auflösung
// std::string image_filename = "fhtw_logo.png";

std::vector<std::string> possible_paths = {
    "test_images/" + image_filename,
    "../test_images/" + image_filename,
    "../../test_images/" + image_filename,
    "../../../test_images/" + image_filename
};

for (const auto& path : possible_paths)
{
    if (std::ifstream(path).good())
    {
        image = cv::imread(path, cv::IMREAD_COLOR);
        image_path = path;
        break;
    }
}

if (image.empty())
{
    std::cerr << "Das Testbild konnte nicht gefunden werden." << std::endl;
}
else
{
    std::cout << "Geladen: " << image_path << std::endl;
    std::cout << "Bildauflösung (H,W,C): "
              << image.rows << ", "
              << image.cols << ", "
              << image.channels() << std::endl;
}

In [ ]:
show_image(image)

## 2. Warum Kantenerkennung?

Ein Kamerabild enthält sehr viele Pixelwerte. Für manche Aufgaben benötigen wir aber nicht die vollständige Bildinformation.

Eine einfache Annahme lautet:

> **Objektgrenzen und starke lokale Helligkeitsänderungen enthalten relevante geometrische Information.**

Eine Kante entspricht einem lokalen Hell-Dunkel- bzw. Dunkel-Hell-Übergang. Mathematisch suchen wir daher nach großen Änderungen der Bildintensität.

Für ein Grauwertbild $f(x,y)$ ist der Gradient

$\nabla f(x,y)=
\begin{bmatrix}
\frac{\partial f}{\partial x}\\
\frac{\partial f}{\partial y}
\end{bmatrix}$

mit der Gradientenstärke

$|\nabla f|=\sqrt{G_x^2+G_y^2}$

und der Gradientenrichtung

$\theta=\operatorname{atan2}(G_y,G_x)$.

Da ein digitales Bild nur an diskreten Pixelpositionen vorliegt, werden die Ableitungen durch Differenzen bzw. entsprechende Filterkerne angenähert.

In [ ]:
cv::Mat gray;
cv::cvtColor(
    image,
    gray,
    cv::COLOR_BGR2GRAY
);

In [ ]:
show_image(gray)

## 3. Sobel: Bildgradient in x- und y-Richtung

Der Sobel-Operator approximiert die partiellen Ableitungen mit zwei unterschiedlichen Kernels.

Für die x-Richtung:

$K_x=
\begin{bmatrix}
-1 & 0 & 1\\
-2 & 0 & 2\\
-1 & 0 & 1
\end{bmatrix}$

und für die y-Richtung:

$K_y=
\begin{bmatrix}
-1 & -2 & -1\\
0 & 0 & 0\\
1 & 2 & 1
\end{bmatrix}$.

Damit berechnen wir

$G_x = K_x * I$

und

$G_y = K_y * I$.

Da bei einer Ableitung auch **negative Werte** auftreten können, verwenden wir für die Berechnung einen Float-Datentyp statt `uint8`.

In [ ]:
cv::Mat gx;
cv::Mat gy;

cv::Sobel(
    gray,
    gx,
    CV_32F,
    1, 0,
    3
);

cv::Sobel(
    gray,
    gy,
    CV_32F,
    0, 1,
    3
);

cv::Mat magnitude;
cv::Mat direction;

cv::magnitude(gx, gy, magnitude);
cv::phase(gx, gy, direction, true);

double gx_min, gx_max;
double gy_min, gy_max;

cv::minMaxLoc(gx, &gx_min, &gx_max);
cv::minMaxLoc(gy, &gy_min, &gy_max);

std::cout << "Gx Wertebereich: "
          << gx_min << " bis " << gx_max << std::endl;

std::cout << "Gy Wertebereich: "
          << gy_min << " bis " << gy_max << std::endl;

### Gradienten visualisieren

Die Gradienten enthalten positive und negative Werte und können daher nicht direkt sinnvoll als normales 8-Bit-Bild dargestellt werden.

Für die **Visualisierung** normalisieren wir die Werte deshalb auf den Bereich 0 bis 255. Die eigentlichen Gradientenwerte bleiben dabei unverändert erhalten.

In [ ]:
cv::Mat gx_vis;
cv::Mat gy_vis;
cv::Mat magnitude_vis;

cv::normalize(
    gx,
    gx_vis,
    0, 255,
    cv::NORM_MINMAX,
    CV_8U
);

cv::normalize(
    gy,
    gy_vis,
    0, 255,
    cv::NORM_MINMAX,
    CV_8U
);

cv::normalize(
    magnitude,
    magnitude_vis,
    0, 255,
    cv::NORM_MINMAX,
    CV_8U
);

**Original**

In [ ]:
show_image(image)

**Gradient $G_x$**

In [ ]:
show_image(gx_vis)

**Gradient $G_y$**

In [ ]:
show_image(gy_vis)

**Gradientenstärke**

In [ ]:
show_image(magnitude_vis)

### Beobachtung

- $G_x$ reagiert auf Änderungen in x-Richtung und hebt daher insbesondere **vertikale Strukturen** hervor.
- $G_y$ reagiert auf Änderungen in y-Richtung und hebt insbesondere **horizontale Strukturen** hervor.
- Die Gradientenstärke kombiniert beide Richtungen.

### Probieren Sie selbst

Ändern Sie die Kernelgröße von Sobel, z.B. auf `1`, `3`, `5` oder `7`.

Welche Auswirkungen sehen Sie auf die Gradienten?

> **Binder-/Notebook-Hinweis:** Die Schreibweise `[](){ ... }()` erzeugt eine kleine anonyme Funktion (Lambda), die direkt ausgeführt wird. Sie wird hier verwendet, damit die Variablen der Übungszelle lokal bleiben und die Zelle nach Änderungen leichter erneut ausgeführt werden kann. In einem normalen lokalen C++-Programm wäre diese zusätzliche Lambda-Konstruktion nicht notwendig.

In [ ]:
[]()
{
    // TODO: Sobel-Kernelgröße verändern
    int sobel_ksize = 5;

    cv::Mat gx_test;
    cv::Mat gy_test;
    cv::Mat magnitude_test;
    cv::Mat magnitude_test_vis;

    cv::Sobel(
        gray,
        gx_test,
        CV_32F,
        1, 0,
        sobel_ksize
    );

    cv::Sobel(
        gray,
        gy_test,
        CV_32F,
        0, 1,
        sobel_ksize
    );

    cv::magnitude(
        gx_test,
        gy_test,
        magnitude_test
    );

    cv::normalize(
        magnitude_test,
        magnitude_test_vis,
        0, 255,
        cv::NORM_MINMAX,
        CV_8U
    );

    std::cout << "Sobel – Kernelgröße "
              << sobel_ksize << std::endl;

    return show_image(magnitude_test_vis);
}()

## 4. Canny Edge Detector

Das reine Sobel-Ergebnis enthält häufig breite oder schwach ausgeprägte Kanten. Der Canny Edge Detector erweitert die Gradientenberechnung um mehrere Verarbeitungsschritte:

1. **Glättung** des Bildes zur Reduktion von Rauschen,
2. Berechnung des Gradienten,
3. **Non-Maximum Suppression**, um breite Kanten auszudünnen,
4. Bewertung der Kanten mit zwei Schwellwerten.

Das Ergebnis ist ein binäres Kantenbild mit möglichst schmalen Kanten.

In [ ]:
cv::Mat gray_blurred;

cv::GaussianBlur(
    gray,
    gray_blurred,
    cv::Size(3, 3),
    0
);

double canny_min = 50;
double canny_max = 150;

cv::Mat edges;

cv::Canny(
    gray_blurred,
    edges,
    canny_min,
    canny_max
);

**Grauwertbild**

In [ ]:
show_image(gray)

**Gaussian Blur**

In [ ]:
show_image(gray_blurred)

**Sobel-Stärke**

In [ ]:
show_image(magnitude_vis)

**Canny**

In [ ]:
show_image(edges)

### Zwei Schwellwerte

Canny verwendet einen unteren und einen oberen Schwellwert.

- sehr starke Kanten werden sicher übernommen,
- sehr schwache Kanten werden verworfen,
- Werte zwischen beiden Schwellwerten werden abhängig von ihrer Verbindung zu stärkeren Kanten behandelt.

### Mini-Aufgabe: Canny-Parameter

Für die Parameteruntersuchung verwenden wir ein anderes Testbild, da im FHTW-Logo fast alle Kanten sehr ähnlich stark ausgeprägt sind. Verwenden Sie entweder das **Lena-** oder das **Messi-Testbild**.

Verändern Sie `canny_min_test` und `canny_max_test`.

Beobachten Sie:

- Wann entstehen zu viele Kanten?
- Wann verschwinden relevante Kanten?
- Welchen Einfluss hat die vorherige Glättung?

In [ ]:
// Testbild auswählen
std::string canny_url =
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/lena.jpg";

// Alternativ: Messi
// std::string canny_url =
//     "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/messi5.jpg";

download_file(canny_url, "testbild.jpg");

cv::Mat image_canny =
    cv::imread("testbild.jpg", cv::IMREAD_COLOR);

if (image_canny.empty())
{
    std::cerr << "Das Canny-Testbild konnte nicht geladen werden."
              << std::endl;
}

cv::Mat gray_canny;

cv::cvtColor(
    image_canny,
    gray_canny,
    cv::COLOR_BGR2GRAY
);

> **Binder-/Notebook-Hinweis:** In der Python-Version werden Originalbild und Ergebnis mit Matplotlib nebeneinander dargestellt. In diesem C++-Notebook werden die Bilder wegen der browserbasierten Rich-Display-Ausgabe nacheinander angezeigt. In einem lokalen C++-Programm könnten dafür einfach zwei `cv::imshow()`-Fenster verwendet werden.

**Testbild**

In [ ]:
show_image(image_canny)

In [ ]:
[]()
{
    // Vor Canny glätten – testen Sie verschiedene Filtergrößen
    cv::Mat gray_canny_blurred;

    cv::GaussianBlur(
        gray_canny,
        gray_canny_blurred,
        cv::Size(5, 5),
        0
    );

    // TODO: Canny-Schwellwerte verändern
    double canny_min_test = 10;
    double canny_max_test = 250;

    cv::Mat edges_test;

    cv::Canny(
        gray_canny_blurred,
        edges_test,
        canny_min_test,
        canny_max_test
    );

    std::cout << "Canny: "
              << canny_min_test << " / "
              << canny_max_test << std::endl;

    return show_image(edges_test);
}()

## 5. Von Kanten zu geometrischen Merkmalen

Ein Kantenbild enthält zunächst nur einzelne Kantenpixel. Für manche robotischen Anwendungen interessieren uns aber explizite geometrische Strukturen wie

- **Linien**,
- **Kreise**,
- oder später komplexere lokale Merkmale.

Die Hough-Transformation sucht solche Strukturen in einem Parameterraum.

Für eine Linie verwenden wir die Darstellung

$\rho = x\cos(\theta)+y\sin(\theta)$

mit

- $\rho$: Abstand der Linie vom Ursprung,
- $\theta$: Winkel des Normalvektors.

Jeder Kantenpunkt stimmt für mehrere Kombinationen aus $\rho$ und $\theta$ ab. Häufen sich viele Stimmen an derselben Stelle im Parameterraum, spricht das für eine Linie im Bild.

## 6. Hough Lines

Für die Liniendetektion verwenden wir das **Sudoku-Testbild** mit vielen gut sichtbaren Linien. Das Bild wird direkt aus den offiziellen OpenCV-Beispieldaten geladen.

In [ ]:
download_file(
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/sudoku.png",
    "hough_lines.png"
);

cv::Mat image_lines =
    cv::imread("hough_lines.png", cv::IMREAD_COLOR);

if (image_lines.empty())
{
    std::cerr << "Das Bild für Hough Lines konnte nicht geladen werden."
              << std::endl;
}

In [ ]:
show_image(image_lines)

In [ ]:
cv::Mat gray_lines;

cv::cvtColor(
    image_lines,
    gray_lines,
    cv::COLOR_BGR2GRAY
);

cv::GaussianBlur(
    gray_lines,
    gray_lines,
    cv::Size(5, 5),
    0
);

cv::Mat edges_lines;

cv::Canny(
    gray_lines,
    edges_lines,
    50,
    150
);

In [ ]:
show_image(edges_lines)

OpenCV liefert für jede gefundene Linie die Parameter $\rho$ und $\theta$. Zur Darstellung werden daraus zwei weit auseinanderliegende Punkte auf der Linie berechnet.

In [ ]:
int hough_threshold = 150;

std::vector<cv::Vec2f> lines;

cv::HoughLines(
    edges_lines,
    lines,
    1,
    CV_PI / 180.0,
    hough_threshold
);

cv::Mat image_hough_lines =
    image_lines.clone();

std::size_t max_lines =
    std::min<std::size_t>(30, lines.size());

for (std::size_t i = 0; i < max_lines; ++i)
{
    float rho = lines[i][0];
    float theta = lines[i][1];

    double a = std::cos(theta);
    double b = std::sin(theta);

    double x0 = a * rho;
    double y0 = b * rho;

    cv::Point pt1(
        cvRound(x0 + 1000 * (-b)),
        cvRound(y0 + 1000 * a)
    );

    cv::Point pt2(
        cvRound(x0 - 1000 * (-b)),
        cvRound(y0 - 1000 * a)
    );

    cv::line(
        image_hough_lines,
        pt1,
        pt2,
        cv::Scalar(0, 0, 255),
        2
    );
}

std::cout << "Gefundene Linien: "
          << lines.size() << std::endl;

In [ ]:
show_image(image_hough_lines)

### Probieren Sie selbst

Der Parameter `threshold` bestimmt, wie viele Stimmen im Hough-Raum für eine Linie erforderlich sind.

Verändern Sie den Wert.

- Was passiert bei einem sehr kleinen Threshold?
- Was passiert bei einem sehr großen Threshold?
- Welche Linien im Bild sind für die Struktur des Sudokus besonders repräsentativ?

> **Binder-/Notebook-Hinweis:** Die Übungszelle wird wieder in einer Lambda-Funktion gekapselt, damit Sie den Threshold verändern und die Zelle mehrfach ausführen können, ohne globale Variablen neu zu deklarieren.

In [ ]:
[]()
{
    // TODO: Hough-Threshold verändern
    int hough_threshold_test = 200;

    std::vector<cv::Vec2f> lines_test;

    cv::HoughLines(
        edges_lines,
        lines_test,
        1,
        CV_PI / 180.0,
        hough_threshold_test
    );

    cv::Mat image_hough_lines_test =
        image_lines.clone();

    std::size_t max_lines =
        std::min<std::size_t>(30, lines_test.size());

    for (std::size_t i = 0; i < max_lines; ++i)
    {
        float rho = lines_test[i][0];
        float theta = lines_test[i][1];

        double a = std::cos(theta);
        double b = std::sin(theta);

        double x0 = a * rho;
        double y0 = b * rho;

        cv::Point pt1(
            cvRound(x0 + 1000 * (-b)),
            cvRound(y0 + 1000 * a)
        );

        cv::Point pt2(
            cvRound(x0 - 1000 * (-b)),
            cvRound(y0 - 1000 * a)
        );

        cv::line(
            image_hough_lines_test,
            pt1,
            pt2,
            cv::Scalar(0, 0, 255),
            2
        );
    }

    std::cout << "Gefundene Linien: "
              << lines_test.size() << std::endl;

    return show_image(image_hough_lines_test);
}()

## 7. Hough Circles

Für Kreise verwenden wir eine andere geometrische Grundstruktur:

$(x-a)^2+(y-b)^2=r^2$

mit

- $(a,b)$: Kreismittelpunkt,
- $r$: Radius.

Die Idee entspricht der Hough-Liniendetektion: Kantenpunkte stimmen für mögliche Kreisparameter ab.

Für dieses erste Beispiel verwenden wir das **Smarties-Testbild** mit mehreren gut sichtbaren kreisförmigen Objekten.

In [ ]:
download_file(
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/smarties.png",
    "hough_circles.png"
);

cv::Mat image_circles =
    cv::imread("hough_circles.png", cv::IMREAD_COLOR);

if (image_circles.empty())
{
    std::cerr << "Das Bild für Hough Circles konnte nicht geladen werden."
              << std::endl;
}

In [ ]:
show_image(image_circles)

In [ ]:
cv::Mat gray_circles;

cv::cvtColor(
    image_circles,
    gray_circles,
    cv::COLOR_BGR2GRAY
);

cv::GaussianBlur(
    gray_circles,
    gray_circles,
    cv::Size(9, 9),
    2
);

std::vector<cv::Vec3f> circles;

cv::HoughCircles(
    gray_circles,
    circles,
    cv::HOUGH_GRADIENT,
    1.2,
    30,
    100,
    30,
    10,
    50
);

cv::Mat image_hough_circles =
    image_circles.clone();

for (const auto& circle : circles)
{
    cv::Point center(
        cvRound(circle[0]),
        cvRound(circle[1])
    );

    int radius =
        cvRound(circle[2]);

    cv::circle(
        image_hough_circles,
        center,
        radius,
        cv::Scalar(0, 255, 0),
        2
    );

    cv::circle(
        image_hough_circles,
        center,
        2,
        cv::Scalar(0, 0, 255),
        3
    );
}

std::cout << "Gefundene Kreise: "
          << circles.size() << std::endl;

In [ ]:
show_image(image_hough_circles)

### Mini-Aufgabe: Parameter der Kreisdetektion

Für die Kreisdetektion verwenden wir `cv::HOUGH_GRADIENT`. Die Funktion besitzt mehrere Parameter, mit denen sowohl die zugrunde liegende Kantendetektion als auch der Suchraum der Hough-Transformation beeinflusst werden können.

- **`dp`** – Auflösung des Hough-Akkumulators relativ zur Bildauflösung.
- **`minDist`** – minimaler Abstand zwischen zwei erkannten Kreismittelpunkten.
- **`param1`** – oberer Schwellwert des intern verwendeten Canny Edge Detectors.
- **`param2`** – Schwellwert des Hough-Akkumulators. Kleinere Werte erkennen mehr, größere Werte nur deutlichere Kreise.
- **`minRadius` / `maxRadius`** – erlaubter Radiusbereich der gesuchten Kreise.

Weitere Informationen zu den Parametern finden Sie in der [OpenCV-4.6-Dokumentation zu `HoughCircles()`](https://docs.opencv.org/4.6.0/dd/d1a/group__imgproc__feature.html#ga47849c3be0d0406ad3ca45db65a25d2d).

Die Parameter beeinflussen sich gegenseitig. Eine gute Kreisdetektion erfordert daher typischerweise sowohl eine geeignete Vorverarbeitung des Bildes als auch sinnvolle Annahmen über Abstand und Größe der erwarteten Kreise.

**Hinweis zum Testbild:** Für die Parameteruntersuchung wird das `board`-Bild verwendet. Es enthält unterschiedlich ausgeprägte kreisförmige Strukturen und ist damit anspruchsvoller als das Smarties-Bild, sodass sich der Einfluss der einzelnen Parameter besser beobachten lässt.

In [ ]:
download_file(
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/board.jpg",
    "board.jpg"
);

cv::Mat image_circles_test =
    cv::imread("board.jpg", cv::IMREAD_COLOR);

if (image_circles_test.empty())
{
    std::cerr << "Das board-Testbild konnte nicht geladen werden."
              << std::endl;
}

cv::Mat gray_circles_test;

cv::cvtColor(
    image_circles_test,
    gray_circles_test,
    cv::COLOR_BGR2GRAY
);

cv::GaussianBlur(
    gray_circles_test,
    gray_circles_test,
    cv::Size(5, 5),
    0
);

> **Binder-/Notebook-Hinweis:** Wie beim Canny-Vergleich werden Originalbild und Ergebnis im C++-Notebook nacheinander angezeigt. Lokal könnten beide Bilder gleichzeitig mit zwei `cv::imshow()`-Fenstern dargestellt werden.

**Original – board.jpg**

In [ ]:
show_image(image_circles_test)

In [ ]:
[]()
{
    // TODO: Parameter der Hough-Kreisdetektion verändern
    int method_test = cv::HOUGH_GRADIENT;

    double dp_test = 1.0;       // Auflösung des Hough-Akkumulators
    double min_dist_test = 25;  // Minimaler Abstand zwischen Kreismittelpunkten
    double param1_test = 100;   // Oberer Threshold des internen Canny Edge Detectors
    double param2_test = 30;    // Threshold des Hough-Akkumulators
    int min_radius_test = 5;    // Minimaler Radius
    int max_radius_test = 25;   // Maximaler Radius

    std::vector<cv::Vec3f> circles_test;

    cv::HoughCircles(
        gray_circles_test,
        circles_test,
        method_test,
        dp_test,
        min_dist_test,
        param1_test,
        param2_test,
        min_radius_test,
        max_radius_test
    );

    cv::Mat image_hough_circles_test =
        image_circles_test.clone();

    for (const auto& circle : circles_test)
    {
        cv::Point center(
            cvRound(circle[0]),
            cvRound(circle[1])
        );

        int radius =
            cvRound(circle[2]);

        cv::circle(
            image_hough_circles_test,
            center,
            radius,
            cv::Scalar(0, 255, 0),
            2
        );

        cv::circle(
            image_hough_circles_test,
            center,
            2,
            cv::Scalar(0, 0, 255),
            3
        );
    }

    std::cout << "Gefundene Kreise: "
              << circles_test.size() << std::endl;

    return show_image(image_hough_circles_test);
}()

## 8. Warum von Kanten zu Bildmerkmalen?

Kanten und einfache geometrische Formen sind nicht für jedes Objekt ausreichend.

Eine Kante beschreibt zwar einen lokalen Übergang, ist entlang ihrer Richtung aber oft nicht eindeutig lokalisierbar. Für komplexere Objekte benötigen wir daher Merkmale, die charakteristischere lokale Bildbereiche beschreiben.

Ein sehr einfacher Ansatz ist **Template Matching**:

> Ein ausgeschnittener Bildbereich dient als Schablone und wird an allen möglichen Positionen mit einem größeren Bild verglichen.

## 9. Template Matching

Sei

- $T$ das Template,
- $I$ das größere Bild.

Bei einer quadratischen Fehlermetrik kann für jede mögliche Position $(x,y)$ beispielsweise

$R(x,y)=
\sum_{x',y'}
\left(
T(x',y')-I(x+x',y+y')
\right)^2$

berechnet werden.

Bei dieser Metrik suchen wir das **Minimum**: Dort ist die Abweichung zwischen Template und Bildausschnitt am kleinsten.

Wir schneiden zunächst einen Bereich direkt aus dem FHTW-Logo aus.

In [ ]:
// Template aus dem realen Testbild ausschneiden
// cv::Rect(x, y, width, height)
cv::Rect template_roi(
    10,
    40,
    155,
    38
);

cv::Mat image_template =
    image(template_roi).clone();

std::cout << "Template-Auflösung (H,W,C): "
          << image_template.rows << ", "
          << image_template.cols << ", "
          << image_template.channels() << std::endl;

In [ ]:
show_image(image_template)

Mit `cv::matchTemplate()` wird das Template über das gesamte Bild geschoben. Wir verwenden `cv::TM_SQDIFF_NORMED`: kleine Werte bedeuten einen guten Match.

In [ ]:
cv::Mat match_result;

cv::matchTemplate(
    image,
    image_template,
    match_result,
    cv::TM_SQDIFF_NORMED
);

double min_val;
double max_val;
cv::Point min_loc;
cv::Point max_loc;

cv::minMaxLoc(
    match_result,
    &min_val,
    &max_val,
    &min_loc,
    &max_loc
);

cv::Point top_left =
    min_loc;

cv::Point bottom_right(
    top_left.x + image_template.cols,
    top_left.y + image_template.rows
);

cv::Mat image_match =
    image.clone();

cv::rectangle(
    image_match,
    top_left,
    bottom_right,
    cv::Scalar(0, 0, 255),
    2
);

std::cout << "Bester Fehlerwert: "
          << min_val << std::endl;

std::cout << "Gefundene Position: ("
          << top_left.x << ", "
          << top_left.y << ")" << std::endl;

// Für die Browserdarstellung wird die Float-Fehlerkarte
// auf ein 8-Bit-Bild normalisiert.
cv::Mat match_result_vis;

cv::normalize(
    match_result,
    match_result_vis,
    0, 255,
    cv::NORM_MINMAX,
    CV_8U
);

cv::Mat match_result_color;

cv::applyColorMap(
    match_result_vis,
    match_result_color,
    cv::COLORMAP_VIRIDIS
);

**Matching-Ergebnis**

In [ ]:
show_image(match_result_color)

**Bester Match**

In [ ]:
show_image(image_match)

### Warum ist Template Matching begrenzt?

Template Matching sucht immer nach der **ähnlichsten Bildposition**. Ein bestes Ergebnis existiert daher auch dann, wenn das gesuchte Objekt gar nicht im Bild vorhanden ist.

Zusätzlich ist eine einzelne Schablone empfindlich gegenüber Veränderungen wie

- Rotation,
- Skalierung,
- Perspektive,
- Beleuchtung,
- Teilverdeckung.

Damit motiviert Template Matching den Übergang zu charakteristischeren lokalen Bildmerkmalen.

Welche Bereiche eignen sich besser als eindeutiges Template – gleichmäßige Flächen oder Bereiche mit charakteristischer Struktur?

## 10. Selbstcheck

Beantworten Sie die Fragen zunächst ohne in die Vorlesungsunterlagen zu schauen.

1. Warum können Kanten eine sinnvolle Reduktion der Bildinformation darstellen?
2. Was beschreiben $G_x$ und $G_y$?
3. Warum sollten Sobel-Gradienten nicht direkt als `uint8` berechnet werden?
4. Welche zusätzlichen Schritte enthält Canny gegenüber einer reinen Sobel-Berechnung?
5. Welche Bedeutung haben $\rho$ und $\theta$ bei Hough Lines?
6. Warum kann Vorwissen über den Radius bei Hough Circles hilfreich sein?
7. Was ist das Ergebnis von Template Matching?
8. Warum ist der beste Template-Match nicht automatisch ein Beweis dafür, dass das Objekt vorhanden ist?
9. Warum benötigen wir für komplexere Objekte charakteristischere lokale Merkmale?

<details>
<summary><b>Kurze Antworten anzeigen</b></summary>

1. Kanten reduzieren ein vollständiges Bild auf lokale Übergänge, die häufig mit Objektgrenzen oder geometrischen Strukturen zusammenhängen.
2. Die lokalen Intensitätsänderungen in x- bzw. y-Richtung.
3. Ableitungen können negative Werte besitzen.
4. Glättung, Gradientenberechnung, Non-Maximum Suppression und eine Bewertung mit zwei Schwellwerten.
5. $\rho$ beschreibt den Abstand der Linie vom Ursprung und $\theta$ die Orientierung ihres Normalvektors.
6. Der Suchraum wird eingeschränkt und Fehlinterpretationen können reduziert werden.
7. Eine Ähnlichkeits- bzw. Fehlerkarte für alle möglichen Positionen des Templates.
8. Es existiert immer eine beste Position, auch wenn das tatsächliche Objekt nicht vorhanden ist.
9. Kanten oder einfache Formen sind bei komplexen Objekten oft nicht eindeutig genug.

</details>

## 11. Take-away

Für die Präsenz-LV sollten Sie folgende Punkte mitnehmen:

- Kanten entsprechen starken lokalen Änderungen der Bildintensität.
- Sobel approximiert den Bildgradienten getrennt in x- und y-Richtung.
- Canny kombiniert Glättung, Gradienteninformation, Kantenausdünnung und Thresholding.
- Die Hough-Transformation überführt Kantenpunkte in einen Parameterraum und kann daraus geometrische Strukturen wie Linien oder Kreise ableiten.
- Template Matching sucht einen bekannten Bildausschnitt durch direkten Vergleich mit vielen Bildpositionen.
- Kanten, Linien, Kreise und Templates funktionieren besonders gut, wenn die gesuchte Struktur bekannt und ausreichend eindeutig ist.
- Für komplexere oder stärker veränderliche Objekte benötigen wir robustere lokale Bildmerkmale.

### Ausblick

In der Vorlesung werden diese Verfahren systematisch eingeordnet. Darauf aufbauend folgt der Übergang von einfachen Kanten und Templates zu charakteristischeren lokalen Features und deren Verwendung für die Objekterkennung.